# Neural Scaling Laws for Jet Tagging

This tutorial reproduces a Chinchilla-style **IsoFLOP scaling-law study** for jet tagging using the [Particle Transformer (ParT)](https://arxiv.org/abs/2202.03772) v3 on the [JetClass](https://arxiv.org/abs/2202.03772) 10-class benchmark.

**Goal:** empirically determine the compute-optimal allocation of a training FLOP budget between model size $N$ (parameters) and dataset size $D$ (training jets):

$$N^*(C) \propto C^a, \qquad D^*(C) \propto C^b, \qquad a + b = 1$$

We use two complementary fitting approaches from [Hoffmann et al. (2022)](https://arxiv.org/abs/2203.15556):

- **Approach 2 (IsoFLOP profiles):** per compute budget $C_k$, fit a quadratic in $\log N$ to the loss-vs-model-size curve, read off $N^*(C_k)$ and $D^*(C_k)$, then fit the power laws.
- **Approach 3 (parametric joint fit):** fit $L(N, D) = E + A/N^\alpha + B/D^\beta$ jointly across all cells.

**Experimental setup:**
- 12 ParT-v3 configurations (S1--S5, M1--M7): width $d \in \{32, 48, 64, 96, 128\}$, depth $L \in \{1, \ldots, 6\}$, params 18k--892k
- 4 IsoFLOP budgets: $C_1 = 3 \times 10^{12}$, $C_2 = 10^{13}$, $C_3 = 3 \times 10^{13}$, $C_4 = 10^{14}$ FLOPs
- Single-pass training, batch size 512, AdamW, warmup+cosine schedule, bf16 AMP
- LR scaling: $\text{lr} = 10^{-3} \times \sqrt{128/d}$
- This yields 35 valid (model, budget) cells out of the $12 \times 4 = 48$ grid

# Preparation

Install the necessary packages:

In [ ]:
! pip install numpy pandas matplotlib scipy

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

%matplotlib inline

# Part 1: Experimental Design

## Model grid

We define 12 Particle Transformer v3 configurations spanning a range of model sizes. Each model is characterised by its embedding dimension $d$, number of transformer layers $L$, and number of attention heads $H$. The key architectural choices are:

- Single embedding layer: `embed_dims = [d]`
- Pairwise embedding: `pair_embed_dims = [d/2, d/2]`
- Pre-activation pairing, global [CLS] token, no CLS layers, no FC head
- ParT v3 with `drop_path_rate = 0`

| Model | d   | L | H | Params  |
|-------|-----|---|---|---------|
| S1    | 32  | 1 | 2 | 17,914  |
| S2    | 32  | 2 | 2 | 34,362  |
| S3    | 48  | 1 | 3 | 39,332  |
| S4    | 48  | 2 | 3 | 76,292  |
| S5    | 64  | 1 | 4 | 69,086  |
| M1    | 64  | 2 | 4 | 134,750 |
| M2    | 64  | 4 | 4 | 266,078 |
| M3    | 96  | 2 | 6 | 301,250 |
| M4    | 64  | 6 | 4 | 397,406 |
| M5    | 128 | 2 | 8 | 533,862 |
| M6    | 96  | 4 | 6 | 596,546 |
| M7    | 96  | 6 | 6 | 891,842 |

## FLOP profiling

We measure per-jet FLOPs for each model using [fvcore](https://github.com/facebookresearch/fvcore). The training FLOPs per jet is estimated as $3\times$ the forward FLOPs (1x forward + 2x backward).

The profiling script is provided at `scaling_laws/flop_profile.py` and the pre-computed results are in `scaling_laws/flops.csv`.

In [ ]:
flops_df = pd.read_csv('scaling_laws/flops.csv')
flops_df

## IsoFLOP grid construction

For each compute budget $C_k$ and model $i$, the number of training jets is:

$$D_{i,k} = \frac{C_k}{\text{train\_flops\_per\_jet}_i}$$

We keep only cells where $10{,}000 \leq D \leq 1{,}000{,}000$ (the training pool contains 1M jets) and the number of optimiser steps is at least 30.

In [ ]:
BUDGETS = {'C1': 3.0e12, 'C2': 1.0e13, 'C3': 3.0e13, 'C4': 1.0e14}
D_MIN, D_MAX = 10_000, 1_000_000
BATCH_SIZE = 512
MIN_STEPS = 30

grid_rows = []
for _, row in flops_df.iterrows():
    for bid, C in BUDGETS.items():
        D = int(C / row['train_flops_per_jet'])
        steps = D // BATCH_SIZE
        if D_MIN <= D <= D_MAX and steps >= MIN_STEPS:
            grid_rows.append({
                'model_id': row['model_id'], 'budget_id': bid,
                'params': row['params'], 'D': D, 'steps': steps,
                'train_flops_per_jet': row['train_flops_per_jet'],
                'C': C,
            })

grid = pd.DataFrame(grid_rows)
print(f'{len(grid)} valid cells out of {len(flops_df) * len(BUDGETS)} total')
grid.head(10)

## Dataset

The study uses the JetClass 10-class dataset (parquet format) with 1M training jets and 1M validation jets:

- **Train:** https://hqu.web.cern.ch/datasets/JetClassMini/train_1M_parquet/
- **Val:** https://hqu.web.cern.ch/datasets/JetClassMini/val_1M_parquet/

Each set contains one file per class (e.g. `HToBB_000.parquet`, `HToBB_120.parquet` for train and val respectively).

The 10 classes are: QCD, $H \to b\bar{b}$, $H \to c\bar{c}$, $H \to gg$, $H \to 4q$, $H \to qq'l\nu$, $Z \to q\bar{q}$, $W \to q\bar{q}'$, $t \to bq\bar{q}'$, $t \to bl\nu$.

## Training

Training is orchestrated by `scaling_laws/launch.py`, which generates and executes [weaver](https://github.com/hqucms/weaver-core) training commands. Key training settings:

- **Single-pass training:** each jet is seen exactly once (1 epoch over $D$ jets)
- **Optimiser:** AdamW with warmup + cosine schedule
- **LR scaling:** $\text{lr} = 10^{-3} \times \sqrt{128 / d}$ (Xavier-style, keeps activation scale stable across widths)
- **Precision:** bf16 AMP

To reproduce the training (requires a GPU and `weaver-core`):
```bash
pip install weaver-core
# Download the dataset to ./JetClassMini/
cd scaling_laws
python launch.py --data-dir ../JetClassMini --run --gpu 0
```

The full sweep of 35 runs takes a few hours on a single GPU. Pre-computed results are provided so you can skip training and jump straight to the analysis.

# Part 2: Analysis

## Load pre-computed results

In [ ]:
df = pd.read_csv('scaling_laws/results.csv')
df['log_N'] = np.log10(df['params'])
df['log_D'] = np.log10(df['D'])
df['log_C'] = np.log10(df['C'])
df['log_L'] = np.log10(df['val_loss'])
df = df.sort_values(['budget_id', 'params']).reset_index(drop=True)
print(f'Loaded {len(df)} cells across {df.budget_id.nunique()} budgets')
df

## Approach 2: IsoFLOP profiles

For each compute budget $C_k$, we plot the validation loss against the number of parameters. At fixed compute, larger models see fewer training jets (since $D = C / \text{FLOPs\_per\_jet}$). There is an optimal model size $N^*(C_k)$ that balances model capacity against data availability.

We fit a quadratic $L(\log N) = a(\log N - \log N^*)^2 + L^*$ to each IsoFLOP curve to locate the minimum. Then we fit $N^*(C) \propto C^a$ and $D^*(C) \propto C^b$ across the budgets.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
budgets = sorted(df['C'].unique())
colors = plt.cm.viridis(np.linspace(0, 0.85, len(budgets)))
optima = []

ax = axes[0]
for C, color in zip(budgets, colors):
    sub = df[df['C'] == C].sort_values('params')
    ax.plot(sub['params'], sub['val_loss'], 'o', color=color, ms=8, label=f'C = {C:.1e}')
    if len(sub) < 3:
        best = sub.loc[sub['val_loss'].idxmin()]
        ax.scatter([best['params']], [best['val_loss']], s=170, color=color,
                   edgecolor='black', lw=1.5, marker='*', zorder=5)
        optima.append({'C': C, 'params_opt': best['params'], 'D_opt': best['D'],
                        'loss_opt': best['val_loss'], 'fit': 'empirical'})
        continue
    logN = np.log10(sub['params'].values)
    a, b, c = np.polyfit(logN, sub['val_loss'].values, 2)
    xs = np.linspace(logN.min() - 0.15, logN.max() + 0.15, 80)
    ys = a * xs**2 + b * xs + c
    ax.plot(10**xs, ys, '--', color=color, lw=1.5)
    if a > 0:
        log_opt = -b / (2 * a)
        N_opt = 10**log_opt
        L_opt = a * log_opt**2 + b * log_opt + c
        fjet_opt = np.interp(log_opt, logN, sub['train_flops_per_jet'].values)
        D_opt = C / fjet_opt
        ax.scatter([N_opt], [L_opt], s=170, color=color, edgecolor='black',
                   lw=1.5, marker='*', zorder=5)
        optima.append({'C': C, 'params_opt': N_opt, 'D_opt': D_opt,
                        'loss_opt': L_opt, 'fit': 'parabola'})
    else:
        best = sub.loc[sub['val_loss'].idxmin()]
        ax.scatter([best['params']], [best['val_loss']], s=170, color=color,
                   edgecolor='red', lw=1.5, marker='*', zorder=5)
        optima.append({'C': C, 'params_opt': best['params'], 'D_opt': best['D'],
                        'loss_opt': best['val_loss'], 'fit': 'inverted'})
ax.set_xscale('log')
ax.set_xlabel('Parameters N')
ax.set_ylabel('Val cross-entropy')
ax.set_title('IsoFLOP curves (Approach 2)')
ax.legend(title='FLOPs')
ax.grid(alpha=0.3)

opt = pd.DataFrame(optima)

# Exclude the lowest compute point (C1) from the power-law fit:
# its parabola minimum extrapolates far below the model range
C_min_for_fit = sorted(budgets)[1]
opt_fit = opt[opt['C'] >= C_min_for_fit]

# N*(C) power law
ax = axes[1]
ax.loglog(opt['C'], opt['params_opt'], 'o', ms=10, color='#2ecc71')
ax.set_xlabel('FLOPs C')
ax.set_ylabel('Optimal N*')
ax.set_title('N*(C) power law')
ax.grid(alpha=0.3, which='both')
if len(opt_fit) >= 2:
    log_C, log_N = np.log10(opt_fit['C']), np.log10(opt_fit['params_opt'])
    a_N, b_N = np.polyfit(log_C, log_N, 1)
    xs = np.logspace(log_C.min() - 0.5, log_C.max() + 0.5, 50)
    ax.plot(xs, 10**(b_N + a_N * np.log10(xs)), 'r--', alpha=0.8,
            label=f'N* $\\propto$ C^{{{a_N:.3f}}}')
    ax.legend()

# D*(C) power law
ax = axes[2]
ax.loglog(opt['C'], opt['D_opt'], 'o', ms=10, color='#e74c3c')
ax.set_xlabel('FLOPs C')
ax.set_ylabel('Optimal D*')
ax.set_title('D*(C) power law')
ax.grid(alpha=0.3, which='both')
if len(opt_fit) >= 2:
    log_D_opt = np.log10(opt_fit['D_opt'])
    a_D, b_D = np.polyfit(log_C, log_D_opt, 1)
    xs = np.logspace(log_C.min() - 0.5, log_C.max() + 0.5, 50)
    ax.plot(xs, 10**(b_D + a_D * np.log10(xs)), 'r--', alpha=0.8,
            label=f'D* $\\propto$ C^{{{a_D:.3f}}}')
    ax.legend()

plt.tight_layout()
plt.show()

print('Optimal points per budget:')
print(opt.to_string(index=False, float_format=lambda x: f'{x:.4g}'))
if len(opt_fit) >= 2:
    print(f'\nPower-law fit (excluding C1):')
    print(f'  N* ~ C^{a_N:.3f}   D* ~ C^{a_D:.3f}   sum = {a_N + a_D:.3f} (should be ~ 1)')

## Approach 3: Parametric joint fit

We fit the three-term loss model from Hoffmann et al.:

$$L(N, D) = E + \frac{A}{N^\alpha} + \frac{B}{D^\beta}$$

where:
- $E$ is the irreducible (Bayes-optimal) loss
- $A/N^\alpha$ captures the power-law decrease in loss with model size
- $B/D^\beta$ captures the power-law decrease in loss with data size

The fit minimises the Huber loss on $\log L_{\text{pred}} - \log L_{\text{obs}}$ (following Chinchilla). We use L-BFGS-B with a multi-start grid over initial conditions.

Given a compute constraint $C \approx 6ND$ (or more precisely, $C = D \times \text{FLOPs\_per\_jet}(N)$), the compute-optimal allocation gives:

$$a = \frac{\beta}{\alpha + \beta}, \qquad b = \frac{\alpha}{\alpha + \beta}$$

so $a + b = 1$ is automatic.

In [ ]:
# Pack data into arrays
N = df['params'].values.astype(float)
D = df['D'].values.astype(float)
L = df['val_loss'].values.astype(float)

ln_N = np.log(N)
ln_D = np.log(D)
ln_L = np.log(L)


def predicted_ln_L(theta, ln_N, ln_D):
    log_A, log_B, E, alpha, beta = theta
    terms = np.stack([
        np.full_like(ln_N, max(E, 1e-12)),
        np.exp(log_A - alpha * ln_N),
        np.exp(log_B - beta * ln_D),
    ], axis=0)
    return np.log(terms.sum(axis=0))


def huber(r, delta=1e-3):
    r = np.asarray(r)
    quad = 0.5 * r**2
    lin = delta * (np.abs(r) - 0.5 * delta)
    return np.where(np.abs(r) <= delta, quad, lin)


def loss_fn(theta, ln_N, ln_D, ln_L, delta=1e-3):
    pred = predicted_ln_L(theta, ln_N, ln_D)
    return huber(pred - ln_L, delta=delta).sum()


# Multi-start grid
best = None
for log_A0 in np.linspace(0, 10, 6):
    for log_B0 in np.linspace(0, 10, 6):
        for E0 in [0.1, 0.5, 1.0, 1.5]:
            for alpha0 in [0.2, 0.4, 0.6]:
                for beta0 in [0.2, 0.4, 0.6]:
                    theta0 = np.array([log_A0, log_B0, E0, alpha0, beta0])
                    try:
                        r = minimize(loss_fn, theta0, args=(ln_N, ln_D, ln_L),
                                     method='L-BFGS-B',
                                     bounds=[(-5, 30), (-5, 30), (1e-3, 5.0),
                                             (0.01, 2.0), (0.01, 2.0)])
                    except Exception:
                        continue
                    if r.success and (best is None or r.fun < best.fun):
                        best = r

log_A, log_B, E, alpha, beta = best.x
A, B = np.exp(log_A), np.exp(log_B)
a_param = beta / (alpha + beta)
b_param = alpha / (alpha + beta)

print('Parametric fit results:')
print(f'  E     = {E:.4f}   (irreducible loss)')
print(f'  A     = {A:.3e}   alpha = {alpha:.3f}')
print(f'  B     = {B:.3e}   beta  = {beta:.3f}')
print(f'\n  Implied compute-optimal exponents:')
print(f'    N*(C) ~ C^{a_param:.3f}')
print(f'    D*(C) ~ C^{b_param:.3f}')
print(f'    a + b = {a_param + b_param:.3f} (1 by construction)')

pred_L = np.exp(predicted_ln_L(best.x, ln_N, ln_D))
resid = pred_L - L
print(f'\n  Residuals (predicted - observed loss):')
print(f'    mean |delta| = {np.mean(np.abs(resid)):.4f}')
print(f'    max  |delta| = {np.max(np.abs(resid)):.4f}')

# Bootstrap uncertainty estimation
N_BOOT = 200
rng = np.random.default_rng(42)
_bounds = [(-5, 30), (-5, 30), (1e-3, 5.0), (0.01, 2.0), (0.01, 2.0)]
boot_params = []
for _ in range(N_BOOT):
    idx = rng.choice(len(N), size=len(N), replace=True)
    r = minimize(loss_fn, best.x, args=(ln_N[idx], ln_D[idx], ln_L[idx]),
                 method='L-BFGS-B', bounds=_bounds)
    if r.success:
        boot_params.append(r.x)
boot_params = np.array(boot_params)

boot_a = boot_params[:, 4] / (boot_params[:, 3] + boot_params[:, 4])
boot_b = boot_params[:, 3] / (boot_params[:, 3] + boot_params[:, 4])

print(f'\nBootstrap uncertainties ({len(boot_params)}/{N_BOOT} converged):')
print(f'  E     = {E:.4f} +/- {boot_params[:, 2].std():.4f}')
print(f'  A     = {A:.3e} +/- {np.exp(boot_params[:, 0]).std():.3e}'
      f'   alpha = {alpha:.3f} +/- {boot_params[:, 3].std():.3f}')
print(f'  B     = {B:.3e} +/- {np.exp(boot_params[:, 1]).std():.3e}'
      f'   beta  = {beta:.3f} +/- {boot_params[:, 4].std():.3f}')
print(f'  N*(C) ~ C^({a_param:.3f} +/- {boot_a.std():.3f})')
print(f'  D*(C) ~ C^({b_param:.3f} +/- {boot_b.std():.3f})')

## Visualise the parametric fit

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Smooth flops_per_jet power law
all_params = df.drop_duplicates('model_id').sort_values('params')
_fjet_slope, _fjet_intercept = np.polyfit(
    np.log10(all_params['params'].values),
    np.log10(all_params['train_flops_per_jet'].values), 1)

def fjet_smooth(N_arr):
    return 10 ** (_fjet_slope * np.log10(N_arr) + _fjet_intercept)

N_grid = np.logspace(np.log10(N.min()) - 0.15, np.log10(N.max()) + 0.15, 200)

# (1) Predicted vs observed loss
ax = axes[0]
ax.plot([L.min() * 0.95, L.max() * 1.05], [L.min() * 0.95, L.max() * 1.05],
        'k--', alpha=0.4, label='y = x')
for C, color in zip(budgets, colors):
    mask = df['C'].values == C
    ax.scatter(L[mask], pred_L[mask], color=color, s=60,
               edgecolor='black', lw=0.5, label=f'{C:.1e}')
ax.set_xlabel('Observed val loss')
ax.set_ylabel('Predicted val loss')
ax.set_title('Parametric fit quality')
ax.grid(alpha=0.3)
ax.legend(title='FLOPs', fontsize=8)

# (2) IsoFLOP frontier with bootstrap bands
ax = axes[1]
for C, color in zip(budgets, colors):
    sub = df[df['C'] == C].sort_values('params')
    D_grid = C / fjet_smooth(N_grid)
    pred_curve = np.exp(predicted_ln_L(best.x, np.log(N_grid), np.log(D_grid)))
    # Bootstrap band
    boot_curves = np.array([
        np.exp(predicted_ln_L(bp, np.log(N_grid), np.log(D_grid)))
        for bp in boot_params])
    lo, hi = np.percentile(boot_curves, [16, 84], axis=0)
    ax.fill_between(N_grid, lo, hi, color=color, alpha=0.15)
    ax.plot(N_grid, pred_curve, '-', color=color, lw=1.5, alpha=0.7)
    ax.plot(sub['params'], sub['val_loss'], 'o', color=color, ms=7,
            label=f'{C:.1e}')
ax.set_xscale('log')
ax.set_xlabel('Parameters N')
ax.set_ylabel('Val loss')
ax.set_title('IsoFLOP slices: parametric vs observed')
ax.grid(alpha=0.3, which='both')
ax.legend(title='FLOPs', fontsize=8)

# (3) Loss surface contour in (N, D) space — tight range around data
ax = axes[2]
Ng = np.logspace(np.log10(N.min()), np.log10(N.max()), 60)
Dg = np.logspace(np.log10(D.min()), np.log10(D.max()), 60)
NN, DD = np.meshgrid(Ng, Dg)
LL = np.exp(predicted_ln_L(best.x, np.log(NN), np.log(DD)))
c = ax.contourf(NN, DD, LL, levels=20, cmap='viridis')
ax.contour(NN, DD, LL, levels=15, colors='white', alpha=0.4, linewidths=0.5)
ax.scatter(N, D, c=L, cmap='viridis', edgecolor='white', s=70,
           vmin=LL.min(), vmax=LL.max())
# IsoFLOP lines for reference
for C, color in zip(budgets, colors):
    N_iso = np.logspace(np.log10(N.min()), np.log10(N.max()), 100)
    D_iso = C / fjet_smooth(N_iso)
    mask = (D_iso >= D.min() * 0.8) & (D_iso <= D.max() * 1.2)
    ax.plot(N_iso[mask], D_iso[mask], '--', color=color, lw=1, alpha=0.5)
# Compute-optimal frontier with bootstrap band
C_frontier = np.logspace(np.log10(3e12 * 0.5), np.log10(1e14 * 2), 50)
N_scan = np.logspace(np.log10(N.min()) - 0.5, np.log10(N.max()) + 0.5, 500)

def _compute_frontier(theta):
    _log_A, _log_B, _E, _al, _be = theta
    _A, _B = np.exp(_log_A), np.exp(_log_B)
    fN, fD = [], []
    for Cf in C_frontier:
        D_scan = Cf / fjet_smooth(N_scan)
        L_scan = _E + _A / N_scan**_al + _B / D_scan**_be
        i_best = np.argmin(L_scan)
        fN.append(N_scan[i_best])
        fD.append(D_scan[i_best])
    return np.array(fN), np.array(fD)

frontier_N, frontier_D = _compute_frontier(best.x)
boot_fN = np.array([_compute_frontier(bp)[0] for bp in boot_params])
boot_fD = np.array([_compute_frontier(bp)[1] for bp in boot_params])
fN_lo, fN_hi = np.percentile(boot_fN, [16, 84], axis=0)
fD_lo, fD_hi = np.percentile(boot_fD, [16, 84], axis=0)
ax.fill_betweenx(frontier_D, fN_lo, fN_hi, color='red', alpha=0.15)
ax.plot(frontier_N, frontier_D, 'r--', lw=2, label='compute-optimal frontier')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Parameters N')
ax.set_ylabel('Training jets D')
ax.set_title('Loss surface + observed cells')
plt.colorbar(c, ax=ax, label='predicted L')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Comparison of approaches

In [ ]:
summary = pd.DataFrame({
    'Method': ['Approach 2 (IsoFLOP)', 'Approach 3 (parametric)'],
    'a (N* ~ C^a)': [f'{a_N:.3f}' if 'a_N' in dir() else 'n/a',
                      f'{a_param:.3f} +/- {boot_a.std():.3f}'],
    'b (D* ~ C^b)': [f'{a_D:.3f}' if 'a_D' in dir() else 'n/a',
                      f'{b_param:.3f} +/- {boot_b.std():.3f}'],
    'a + b': [f'{a_N + a_D:.3f}' if 'a_N' in dir() else 'n/a',
              f'{a_param + b_param:.3f}'],
})
summary

# Summary

From the parametric fit (Approach 3), we find:

- $N^*(C) \propto C^{0.473}$, $D^*(C) \propto C^{0.527}$ -- both model size and data should scale roughly equally with compute
- Irreducible loss $E \approx 0.28$ (underestimated; the true Bayes error is likely around 0.46 based on the nominal ParT accuracy of 0.861 on JetClass)
- The optimal data-to-parameter ratio is $D/N \approx 11$ jets per parameter
- Compared to Chinchilla's LLM finding ($D/N \approx 20$ tokens/param), jet tagging is somewhat more parameter-hungry, possibly because the classification task extracts less information per sample than next-token prediction

**Key takeaway:** When budgeting compute for jet taggers, scaling the model and data roughly equally (both as $\sim C^{0.5}$) is near-optimal. This is consistent with the Chinchilla finding for LLMs, suggesting that the "equal scaling" principle generalises beyond language models.